In [3]:
from simulations.simulations import path_wise_dataset_1, treatment_col_index, confounder_col_index, mediator1_col_index, mediator2_col_index, ModelWrapper, calculate_true_cate_but_mediator2
import numpy as np

np.random.seed(0)
from sklearn.experimental import enable_iterative_imputer

In [4]:

from re import L
import numpy as np
import pandas as pd
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LogisticRegression
import time
from sklearn.linear_model import LogisticRegression
from simulations.simulations import ModelWrapper


def model_outcomes_coal(model, coal, local_obs, ref_point, imp=None, predict_proba=False, seed=0):
    
    start = time.time()
    
    ## this function outputs a list of all model predictions for a single coalition
    # the input is the coalition of interest as a binary vector eg. [1,1,0,1]
    # ref point is the entire reference distirbution (here we take x_train)
    # local_obs is the instance we aim to explain
    
    n_obs, d_obs = np.shape(local_obs)

    # we consider all 2**n coalitions and all n reference points
    # get constants
    
    n, d = np.shape(ref_point)
    assert d == d_obs
    
    # train imputation algorithm for conditional references   
    # create an "all_imputed_coalitions" 3D matrix with all "artificial inputs" we get by imputing
    coalitions = np.array([[int(i) for i in '0'*((d)-len(bin(j))+2) + bin(j)[2:]] for j in range(2**d)]) 
    all_imputed_coalitions = np.zeros((n_obs, n, d))

    
     ## below we build the concatenated inputs using conditional imputation for dropped features and averages
    for k in range(n):#reference point index
        vect=(1-coal) 
        vect = vect.astype('float')
        vect[vect == 1] = 'nan'    # vect is a binary vector of being "absent" ('nan') or "present" (1)
        # impute conditionally with Bayesian Ridge MICE 
        imputed_coalitions = coal * local_obs + vect #either local obs value or 'nan'
        #print(imputed_coalitions)
        nans = (np.isnan(imputed_coalitions).sum(axis=0) > 0)
        if (~nans).sum() == 0 or imp is None:
            idx = np.random.randint(n, size=n_obs)
            imputation = ref_point[idx]
            imputed_coalitions[:,nans] = imputation[:,nans]
        else:
            imputed_coalitions = imp.transform(imputed_coalitions)       
        all_imputed_coalitions[:, k, :] = imputed_coalitions
        
    end = time.time()
    print('Execution time (s) :', end - start)
    #we ultimately return the model predictions
    if predict_proba:
        return model.predict_proba(all_imputed_coalitions.reshape(n_obs * n, d))[:,1].reshape(n_obs, n)
    else:
        return model.predict(all_imputed_coalitions.reshape(n_obs * n, d)).reshape(n_obs, n)


class IterativeImputerSubsets(object):

    def __init__(self, subsets=[], **kwargs):

        self.imps = []
        self.subsets = subsets
        for subset in self.subsets:
            self.imps.append(IterativeImputer(**kwargs))

    def fit(self, x_train):
        for imp, subset in zip(self.imps, self.subsets):
            x_train_subset = x_train[:, subset]
            imp.fit(x_train_subset)

    def transform(self, x):
        x_new = x
        for imp, subset in zip(self.imps, self.subsets):
            x_new[:, subset] = imp.transform(x[:, subset])
        return x_new

psis_specific_q_ratios = []
psis_specific_d_ratios = []
phi_direct_cs_ratios = []
phi_indirect_cs_ratios = []



In [5]:

def compute_effects(local_obs, outcome, data, y, cate_only_mode=False):

    start = time.time()

    # p = 0.5 * np.ones(N)
    # t = np.random.binomial(n=1, p=p)

    # q = np.random.uniform(size=N)
    # q = t * 3 / 5 * q + (1 - t) * (3 / 5 * q + 2 / 5)
    # d = np.random.binomial(n=1, p=4 / 5 - 3 / 5 * t)

    # loc=5
    # scale=10
    # (coef_t, coef_q, coef_d, coef_t_q, coef_t_d) = tuple(np.random.normal(loc=loc, scale=scale, size=5))
    # y = coef_t*t + coef_q*q + coef_d*d + coef_t_q*t*q + coef_t_d*t*d

    x = pd.DataFrame({'t': data[:, 0], 'c': data[:, 1], 'q': data[:, 2], 'd': data[:, 3]})
    prop_train = 0.3
    N_train = int(prop_train * data.shape[0])
    x_train = x.iloc[:N_train]
    y_train = y[:N_train]
    x_test = x.iloc[N_train:]
    y_test = y[N_train:]
    imp = IterativeImputer(max_iter=100, random_state=0, sample_posterior=True)
    imp.fit(x.values) #imputer learns from marginal distribution


    imp_all_but_t = IterativeImputer(max_iter=100, random_state=0, sample_posterior=True)
    imp_all_but_t.fit(x_train[['c','q','d']].values)

    propensity = LogisticRegression()
    propensity.fit(x_train[['c','q','d']], x_train['t'])

    local_obs_propensity = [local_obs[0][1:]]


    print('outcomes_all_but_d')
    outcomes_all_but_d = model_outcomes_coal(model=outcome, coal=np.array([1,1,1,0]), local_obs=local_obs, ref_point=x_train.values, imp=imp)
    print('outcomes_all_but_d_t')
    outcomes_all_but_d_t = model_outcomes_coal(model=outcome, coal=np.array([0,1,1,0]), local_obs=local_obs, ref_point=x_train.values, imp=imp)
    print('propensities_all_but_d')
    propensities_all_but_d =  model_outcomes_coal(model=propensity, coal=np.array([1,1,0]), local_obs=local_obs_propensity, ref_point=x_train[['c','q','d']].values, imp=imp_all_but_t, predict_proba=True)


    print("outcomes_all_but_d.mean(axis=1)")
    print(outcomes_all_but_d.mean(axis=1))

    print("outcomes_all_but_d_t.mean(axis=1)")
    print(outcomes_all_but_d_t.mean(axis=1))


    psis_coal_all_but_d = (outcomes_all_but_d.mean(axis=1) - outcomes_all_but_d_t.mean(axis=1)) / (local_obs[0][0] - propensities_all_but_d.mean(axis=1))


    print("propensities_all_but_d.mean(axis=1)")
    print(propensities_all_but_d.mean(axis=1))

    if cate_only_mode:
        return {
            "path_wise_shap_t_m2_y": 0,
            "path_wise_shap_t_m1_y": 0,   
            "cate_mediator1_mediator2_confounder": 0,
            "cate_mediator1_confounder": psis_coal_all_but_d,
            "cate_mediator2_confounder": 0,
        }


    print('outcomes_all')
    outcomes_all = model_outcomes_coal(model=outcome, coal=np.array([1,1,1,1]), local_obs=local_obs, ref_point=x_train.values, imp=imp)
    print('outcomes_all_but_t')
    outcomes_all_but_t = model_outcomes_coal(model=outcome, coal=np.array([1,1,1,0]), local_obs=local_obs, ref_point=x_train.values, imp=imp)
    print('propensities_all')
    propensities_all =  model_outcomes_coal(model=propensity, coal=np.array([1,1,1]), local_obs=local_obs_propensity, ref_point=x_train[['c','q','d']].values, imp=imp_all_but_t, predict_proba=True)

    print(propensities_all)
    
    psis_coal_all = (outcomes_all.mean(axis=1) - outcomes_all_but_t.mean(axis=1)) / (local_obs[0][0] - propensities_all.mean(axis=1))
    
    psis_specific_d = psis_coal_all - psis_coal_all_but_d



    print("___________")
    print(psis_coal_all)

    print('outcomes_all_but_q')
    outcomes_all_but_q = model_outcomes_coal(model=outcome, coal=np.array([1, 1,0,1]), local_obs=local_obs, ref_point=x_train.values, imp=imp)
    print('outcomes_all_but_q_t')
    outcomes_all_but_q_t = model_outcomes_coal(model=outcome, coal=np.array([0, 1,0,1]), local_obs=local_obs, ref_point=x_train.values, imp=imp)
    print('propensities_all_but_q')
    propensities_all_but_q =  model_outcomes_coal(model=propensity, coal=np.array([1,0,1]), local_obs=local_obs_propensity, ref_point=x_train[['c','q','d']].values, imp=imp_all_but_t, predict_proba=True)

    psis_coal_all_but_q = (outcomes_all_but_q.mean(axis=1) - outcomes_all_but_q_t.mean(axis=1)) / (local_obs[0][0] - propensities_all_but_q.mean(axis=1))

    psis_specific_q = psis_coal_all - psis_coal_all_but_q

    print('psis_specific_q_ratio: ', np.abs(psis_specific_q))
    psis_specific_q_ratios.append(np.abs(psis_specific_q))

    print('psis_specific_d_ratio: ', np.abs(psis_specific_d))
    psis_specific_d_ratios.append(np.abs(psis_specific_d))

    return {
            "path_wise_shap_t_m2_y": psis_specific_d,
            "path_wise_shap_t_m1_y": psis_specific_q,
            "cate_mediator1_mediator2_confounder": psis_coal_all,
            "cate_mediator1_confounder": psis_coal_all_but_d,
            "cate_mediator2_confounder": psis_coal_all_but_q,
    }   

In [6]:
X_test1, y_test1 = path_wise_dataset_1(num_samples=1000)

compute_effects([[1, 0.2, 0.6, 1]], ModelWrapper(), X_test1, y_test1, cate_only_mode=True)

outcomes_all_but_d
Execution time (s) : 11.542139053344727
outcomes_all_but_d_t
Execution time (s) : 21.360275268554688
propensities_all_but_d
Execution time (s) : 11.45119309425354
outcomes_all_but_d.mean(axis=1)
[1.11552459]
outcomes_all_but_d_t.mean(axis=1)
[0.73358676]
propensities_all_but_d.mean(axis=1)
[0.70174081]


/Users/maksym/Documents/Files/Heavy/Work/NYU_NEW/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


{'path_wise_shap_t_m2_y': 0,
 'path_wise_shap_t_m1_y': 0,
 'cate_mediator1_mediator2_confounder': 0,
 'cate_mediator1_confounder': array([1.28055682]),
 'cate_mediator2_confounder': 0}

In [7]:
def monte_carlo_evaluation_all_models_path_wise_approach(sample, model_types=None, 
                                                  num_runs=100, num_samples=1000, base_seed=42):
    """
    Perform Monte Carlo evaluation for all specified model types using doubly robust estimator.
    
    Args:
        sample: The sample for which to evaluate CATE
        model_types: List of model types to test
        num_runs: Number of Monte Carlo runs
        num_samples: Number of samples per run for training the estimator
        base_seed: Base seed for reproducibility (each run gets base_seed + run_number)
    
    Returns:
        Dictionary with results for each model type
    """
    results = {}
    true_value = calculate_true_cate_but_mediator2(sample)
    
    print(f"True CATE without mediator2: {true_value:.4f}")
    print(f"Testing {len(model_types)} model types: {model_types}")
    print(f"Running {num_runs} Monte Carlo simulations for each model...")
    
    for model_type, propensity_model_type in model_types:
        print(f"\n{'='*60}")
        print(f"EVALUATING MODEL: {model_type}")
        print(f"{'='*60}")
        
        estimates = []
        
        for run in range(num_runs):
            # Generate new dataset for this run
            X_test, y_test = path_wise_dataset_1(num_samples=num_samples, seed=base_seed+run)
            # Get estimate from the doubly robust estimator with specific model type
            result = compute_effects(
                [sample], 
                ModelWrapper(), 
                X_test, 
                y_test,
                cate_only_mode=True
            )

            estimate = result["cate_mediator1_confounder"]  # Extract scalar value
            print("@@@@@@@@@@@@@@@@@@")
            print(true_value)
            print("@@@@@@@@@@@@@@@@@@")
            print(estimate)
            print("@@@@@@@@@@@@@@@@@@")

            if estimate == np.nan:
                raise Exception("Failed to indetify")
            estimates.append(estimate)
            
            if (run + 1) % 20 == 0:
                print(f"  {model_type}_{propensity_model_type}: Completed {run + 1}/{num_runs} runs")
        
        estimates = np.array(estimates)
        
        # Calculate only MAE and Monte Carlo Error
        errors = estimates - true_value
        mae = np.mean(np.abs(errors))
        
        # Monte Carlo Error (standard error of the mean)
        monte_carlo_error = np.std(estimates) / np.sqrt(num_runs)
        
        results[f"{model_type}_{propensity_model_type}"] = {
            'true_value': true_value,
            'estimates': estimates,
            'mae': mae,
            'monte_carlo_error': monte_carlo_error
        }
        
        print(f"  {model_type}_{propensity_model_type} Results:")
        print(f"    MAE: {mae:.4f}")
        print(f"    Monte Carlo Error: {monte_carlo_error:.4f}")
    
    return results


In [8]:
def evaluate_multiple_samples_averaged_doubly_robust(samples, model_types=None, 
                                                   num_runs=5, num_samples=10000, base_seed=42):
    """
    Evaluate multiple samples and return averaged results across all samples for doubly robust estimator.

    Args:
        samples: List of samples to evaluate
        model_types: List of model types to test
        num_runs: Number of Monte Carlo runs per sample
        num_samples: Number of samples per run
        base_seed: Base seed for reproducibility

    Returns:
        Dictionary with averaged results and individual sample results
    """
    import pandas as pd
    import numpy as np
    
    all_results = {}
    sample_results = {}
    
    print(f"Evaluating {len(samples)} samples with {len(model_types)} model types each (Doubly Robust)")
    print(f"Total evaluations: {len(samples)} × {len(model_types)} = {len(samples) * len(model_types)}")
    print("="*80)
    
    # Evaluate each sample
    for i, sample in enumerate(samples):
        print(f"\nSAMPLE {i+1}: {sample}")
        print(f"True CATE: {calculate_true_cate_but_mediator2(sample):.4f}")
        print("-" * 60)
        
        # Run evaluation for this sample
        sample_result = monte_carlo_evaluation_all_models_path_wise_approach(
            sample, 
            model_types, 
            num_runs, 
            num_samples, 
            base_seed + i  # Different seed for each sample
        )

        sample_results[f"sample_{i+1}"] = sample_result
        
        # Print results for this sample
        for model_type, propensity_model_type in model_types:
            mae = sample_result[f"{model_type}_{propensity_model_type}"]['mae']
            mce = sample_result[f"{model_type}_{propensity_model_type}"]['monte_carlo_error']
            print(f"  {model_type}_{propensity_model_type}: MAE={mae:.4f}, MCE={mce:.4f}")
    
    # Calculate averaged results
    print("\n" + "="*80)
    print("AVERAGED RESULTS ACROSS ALL SAMPLES (DOUBLY ROBUST)")
    print("="*80)
    
    averaged_results = {}
    for model_type, propensity_model_type in model_types:
        # Collect all estimates and true values for this model type across samples
        all_estimates = []
        all_true_values = []
        
        for i in range(len(samples)):
            estimates = sample_results[f"sample_{i+1}"][f"{model_type}_{propensity_model_type}"]['estimates']
            true_value = sample_results[f"sample_{i+1}"][f"{model_type}_{propensity_model_type}"]['true_value']
            all_estimates.extend(estimates)
            all_true_values.extend([true_value] * len(estimates))
        
        # Compute MAE for all samples combined
        all_estimates = np.array(all_estimates)
        all_true_values = np.array(all_true_values)
        combined_mae = np.mean(np.abs(all_estimates - all_true_values))
        
        # Compute Monte Carlo error for all samples combined
        # Monte Carlo error is the standard error of the mean: std(estimates) / sqrt(n)
        combined_mce = np.std(all_estimates) / np.sqrt(len(all_estimates))
        
        averaged_results[f"{model_type}_{propensity_model_type}"] = {
            'mae_combined': combined_mae,
            'mce_combined': combined_mce,
            'total_estimates': len(all_estimates)
        }
    
    # Create comparison table
    comparison_data = []
    for model_type, propensity_model_type in model_types:
        result = averaged_results[f"{model_type}_{propensity_model_type}"]
        comparison_data.append({
            'Model': f"{model_type}_{propensity_model_type}",
            'MAE (Combined)': f"{result['mae_combined']:.4f}",
            'Monte Carlo Error (Combined)': f"{result['mce_combined']:.4f}",
            'Total Estimates': result['total_estimates']
        })
    
    df_comparison = pd.DataFrame(comparison_data)
    print(df_comparison.to_string(index=False))
    
    # Find best performing model by combined MAE
    best_model = min(model_types, key=lambda x: averaged_results[f"{x[0]}_{x[1]}"]['mae_combined'])
    print(f"\nBest performing model (by combined MAE): {best_model[0]}_{best_model[1]}")
    print(f"Combined MAE: {averaged_results[f"{best_model[0]}_{best_model[1]}"]['mae_combined']:.4f}")
    print(f"Total estimates used: {averaged_results[f"{best_model[0]}_{best_model[1]}"]['total_estimates']}")
    
    return averaged_results, sample_results, df_comparison


In [11]:
# Test with 3 different samples and compute averaged results for doubly robust estimator
samples = [
    [1, 0.2, 0.3, 1],
    [1, 0.2, 0.4, 1],
    [1, 0.2, 0.5, 1],
    [1, 0.2, 0.6, 1],
    [1, 0.2, 0.7, 1],
]

print("Testing 3 samples with all model types and computing averaged results (Doubly Robust)...")
print("="*80)

# Run comprehensive evaluation for all samples
averaged_results, sample_results, comparison_df = evaluate_multiple_samples_averaged_doubly_robust(
    samples, 
    model_types=[("any", "any")],
    num_runs=10,
    num_samples=1000,
    base_seed=0
)


Testing 3 samples with all model types and computing averaged results (Doubly Robust)...
Evaluating 5 samples with 1 model types each (Doubly Robust)
Total evaluations: 5 × 1 = 5

SAMPLE 1: [1, 0.2, 0.3, 1]
True CATE: 1.1930
------------------------------------------------------------
True CATE without mediator2: 1.1930
Testing 1 model types: [('any', 'any')]
Running 10 Monte Carlo simulations for each model...

EVALUATING MODEL: any
outcomes_all_but_d
Execution time (s) : 13.608947992324829
outcomes_all_but_d_t
Execution time (s) : 21.898927211761475
propensities_all_but_d
Execution time (s) : 11.240002155303955
outcomes_all_but_d.mean(axis=1)
[1.47939603]
outcomes_all_but_d_t.mean(axis=1)
[0.96167764]
propensities_all_but_d.mean(axis=1)
[0.61529195]
@@@@@@@@@@@@@@@@@@
1.193
@@@@@@@@@@@@@@@@@@
[1.34574358]
@@@@@@@@@@@@@@@@@@


/Users/maksym/Documents/Files/Heavy/Work/NYU_NEW/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


outcomes_all_but_d
Execution time (s) : 11.65187406539917
outcomes_all_but_d_t
Execution time (s) : 22.58021306991577
propensities_all_but_d
Execution time (s) : 11.980210781097412
outcomes_all_but_d.mean(axis=1)
[1.41968569]
outcomes_all_but_d_t.mean(axis=1)
[0.91791596]
propensities_all_but_d.mean(axis=1)
[0.56332209]
@@@@@@@@@@@@@@@@@@
1.193
@@@@@@@@@@@@@@@@@@
[1.14906139]
@@@@@@@@@@@@@@@@@@


/Users/maksym/Documents/Files/Heavy/Work/NYU_NEW/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


outcomes_all_but_d
Execution time (s) : 11.894316911697388
outcomes_all_but_d_t
Execution time (s) : 22.681375980377197
propensities_all_but_d
Execution time (s) : 11.394752025604248
outcomes_all_but_d.mean(axis=1)
[1.45758212]
outcomes_all_but_d_t.mean(axis=1)
[0.94714639]
propensities_all_but_d.mean(axis=1)
[0.62445349]
@@@@@@@@@@@@@@@@@@
1.193
@@@@@@@@@@@@@@@@@@
[1.35918112]
@@@@@@@@@@@@@@@@@@


...

/Users/maksym/Documents/Files/Heavy/Work/NYU_NEW/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


outcomes_all_but_d
Execution time (s) : 11.37397575378418
outcomes_all_but_d_t
Execution time (s) : 21.889163970947266
propensities_all_but_d
Execution time (s) : 11.062670946121216
outcomes_all_but_d.mean(axis=1)
[0.92529919]
outcomes_all_but_d_t.mean(axis=1)
[0.60000375]
propensities_all_but_d.mean(axis=1)
[0.77393268]
@@@@@@@@@@@@@@@@@@
0.9930000000000001
@@@@@@@@@@@@@@@@@@
[1.43893173]
@@@@@@@@@@@@@@@@@@


/Users/maksym/Documents/Files/Heavy/Work/NYU_NEW/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


outcomes_all_but_d
Execution time (s) : 10.91623306274414
outcomes_all_but_d_t
Execution time (s) : 22.041145086288452
propensities_all_but_d
Execution time (s) : 11.286768198013306
outcomes_all_but_d.mean(axis=1)
[0.9667625]
outcomes_all_but_d_t.mean(axis=1)
[0.65730598]
propensities_all_but_d.mean(axis=1)
[0.82321025]
@@@@@@@@@@@@@@@@@@
0.9930000000000001
@@@@@@@@@@@@@@@@@@
[1.75042115]
@@@@@@@@@@@@@@@@@@
  any_any Results:
    MAE: 0.3164
    Monte Carlo Error: 0.0581
  any_any: MAE=0.3164, MCE=0.0581

AVERAGED RESULTS ACROSS ALL SAMPLES (DOUBLY ROBUST)
  Model MAE (Combined) Monte Carlo Error (Combined)  Total Estimates
any_any         0.1912                       0.0162               50

Best performing model (by combined MAE): any_any
Combined MAE: 0.1912
Total estimates used: 50


/Users/maksym/Documents/Files/Heavy/Work/NYU_NEW/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
